# Jev gateway on a Colab GPU
Try a typed-decision classifier on a free GPU: load a model, ask it questions with text, an image or video frames, and see how fast it answers.

**How to use: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.** That is the whole setup. It installs what it needs, downloads
the model (a few minutes the first time, with progress shown) and opens a control panel.

**Every cell is safe to run again.** Stuck or something looks wrong? Run the cell again; it picks up where it is. If that does not help,
run the **Reset** cell at the bottom, then Run all.

* **Disk is budgeted.** Colab's disk is small, so only the last two models used stay downloaded; loading a third deletes the oldest first.
* **Interactive by default.** The public URL and the keep-alive audio (step 6) are opt-in; read the note there first.

## 1. Set up
These cells write the gateway's source files into the Colab machine. You do not need to read or change them. To add a model, add one entry to `models.json`.

In [ ]:
!mkdir -p /content/gw/jevgw/ui /content/work

In [ ]:
%%writefile /content/gw/jevgw/__init__.py
"""jevgw: one decision model at a time behind one TypeSafe-compatible /v1/systemone endpoint."""

__version__ = "0.2.0"

In [ ]:
%%writefile /content/gw/jevgw/catalog.py
"""The model catalog (models.json) and the rule for what fits a GPU."""

from __future__ import annotations

import json
import subprocess
from pathlib import Path

KINDS = {"llama", "proc", "pending", "none"}
RUNNABLE = {"llama", "proc"}
HEADROOM_MIB = 800  # CUDA context and the desktop's share: a model must fit with this much to spare
_REQUIRED = {"llama": ("repo", "file", "file_gib"), "proc": ("cmd",)}


class CatalogError(ValueError):
    """models.json is malformed."""


def load(path: str | Path) -> dict[str, dict]:
    """Read and validate the catalog. Returns {id: entry}, in file order."""
    models: dict[str, dict] = {}
    for entry in json.loads(Path(path).read_text())["models"]:
        for key in ("id", "name", "kind"):
            if key not in entry:
                raise CatalogError(f"entry {entry.get('id', entry)!r} is missing {key!r}")
        if entry["kind"] not in KINDS:
            raise CatalogError(f"{entry['id']}: kind {entry['kind']!r} is not one of {sorted(KINDS)}")
        for key in _REQUIRED.get(entry["kind"], ()):
            if key not in entry:
                raise CatalogError(f"{entry['id']}: kind {entry['kind']!r} needs {key!r}")
        if entry.get("mmproj") and "mmproj_gib" not in entry:
            raise CatalogError(f"{entry['id']}: an mmproj needs 'mmproj_gib' (its size, for the disk budget)")
        if entry["id"] in models:
            raise CatalogError(f"duplicate id {entry['id']!r}")
        models[entry["id"]] = entry
    return models


def runnable(entry: dict) -> bool:
    return entry["kind"] in RUNNABLE


def fits(entry: dict, gpu_mib: int) -> bool:
    """True if the model's peak GPU memory fits with headroom. A model that needs none (CPU) always fits."""
    need = entry.get("vram_gib", 0) * 1024
    return need == 0 or need + HEADROOM_MIB <= gpu_mib


def gpu_info() -> tuple[str | None, int]:
    """(name, total MiB) of GPU 0, or (None, 0) on a CPU-only machine."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True, timeout=10
        )
        name, mib = out.strip().splitlines()[0].rsplit(",", 1)
        return name.strip(), int(mib)
    except (OSError, subprocess.SubprocessError, ValueError, IndexError):
        return None, 0

In [ ]:
%%writefile /content/gw/jevgw/disk.py
"""Disk budget for downloaded model files.

A Colab session has far less disk than a server, and a GGUF is several GiB, so downloads are budgeted: at most `keep` models stay on
disk and a margin of free space is always left. Before a model is downloaded, the least recently used ones are deleted until it fits.
Files are shared between models (both Clef quants use one vision projector), so a file is deleted only when no other cached model,
and not the one about to load, needs it.
"""

from __future__ import annotations

import json
import shutil
from pathlib import Path

GIB = 2**30


class DiskFull(Exception):
    """The model cannot be made to fit on disk, even after deleting every other cached model."""


def files_of(entry: dict) -> dict[str, float]:
    """{file name: expected GiB} for what a model needs on disk. Empty for models that bring their own install."""
    if entry.get("kind") != "llama":
        return {}
    files = {entry["file"]: entry.get("file_gib", 0.0)}
    if entry.get("mmproj"):
        files[entry["mmproj"]] = entry.get("mmproj_gib", 0.0)
    return files


class Disk:
    def __init__(
        self, folder: str | Path, state_file: str | Path, keep: int = 2, margin_gib: float = 5.0, usage=shutil.disk_usage
    ):
        self.folder, self.state_file = Path(folder), Path(state_file)
        self.keep, self.margin = keep, margin_gib * GIB
        self._usage = usage
        self.folder.mkdir(parents=True, exist_ok=True)

    # -- what is on disk ---------------------------------------------------------------------------------------------

    def cached(self, entry: dict) -> bool:
        names = files_of(entry)
        return bool(names) and all((self.folder / name).exists() for name in names)

    def _missing_bytes(self, entry: dict) -> float:
        return sum(gib * GIB for name, gib in files_of(entry).items() if not (self.folder / name).exists())

    def _stamps(self) -> dict[str, float]:
        try:
            return json.loads(self.state_file.read_text())
        except (OSError, ValueError):
            return {}

    def touch(self, model_id: str, now: float) -> None:
        """Record that a model was just used (it is then the last to be deleted)."""
        stamps = self._stamps()
        stamps[model_id] = now
        self.state_file.write_text(json.dumps(stamps))

    def downloaded_gib(self, entry: dict) -> tuple[float, float]:
        """(GiB on disk so far, GiB expected) for a model, counting a download in flight.

        Hugging Face names a partial file after a hash, not the model file, so every `.incomplete` file counts: the gateway loads one
        model at a time and downloads its files one after another.
        """
        partial = self.folder / ".cache" / "huggingface" / "download"
        done = sum(f.stat().st_size for f in partial.glob("*.incomplete"))
        done += sum((self.folder / name).stat().st_size for name in files_of(entry) if (self.folder / name).exists())
        return done / GIB, sum(files_of(entry).values())

    def free_gib(self) -> float:
        return self._usage(self.folder).free / GIB

    def status(self, models: dict[str, dict]) -> dict:
        usage = self._usage(self.folder)
        return {
            "free_gib": round(usage.free / GIB, 1),
            "total_gib": round(usage.total / GIB, 1),
            "keep": self.keep,
            "cached": [mid for mid, entry in models.items() if self.cached(entry)],
        }

    # -- making room -------------------------------------------------------------------------------------------------

    def make_room(self, target: dict, models: dict[str, dict]) -> list[str]:
        """Delete least recently used cached models until `target` fits within `keep` and the free-space margin.

        Returns the ids deleted. Raises DiskFull if it still cannot fit.
        """
        need = self._missing_bytes(target)
        stamps = self._stamps()
        others = sorted(
            (mid for mid, e in models.items() if e is not target and self.cached(e)), key=lambda mid: stamps.get(mid, 0)
        )
        evicted: list[str] = []
        while True:
            fits_count = len(others) + 1 <= self.keep
            fits_space = need == 0 or self._usage(self.folder).free - need >= self.margin  # nothing to download, nothing to fit
            if fits_count and fits_space:
                return evicted
            if not others:
                raise DiskFull(
                    f"{target['id']} needs {need / GIB:.1f} GiB more disk, with {self.margin / GIB:.0f} GiB kept free; "
                    f"{self.free_gib():.1f} GiB is free after deleting every other model"
                )
            victim = others.pop(0)
            self.delete(victim, models, keep_for=target)
            evicted.append(victim)

    def delete(self, model_id: str, models: dict[str, dict], keep_for: dict | None = None) -> None:
        """Delete a model's files, except any still needed by `keep_for` or by another cached model."""
        needed: set[str] = set(files_of(keep_for)) if keep_for else set()
        for other_id, other in models.items():
            if other_id != model_id and self.cached(other):
                needed |= set(files_of(other))
        for name in files_of(models[model_id]):
            if name not in needed:
                (self.folder / name).unlink(missing_ok=True)

In [ ]:
%%writefile /content/gw/jevgw/backends.py
"""How a model is served: a child process that speaks /v1/systemone on a local port.

Two kinds. "llama" runs llama.cpp's llama-server on a GGUF file. "proc" runs a recipe from models.json:
setup commands once, then a launch command. Both are started, probed until healthy, proxied to, and stopped here.
"""

from __future__ import annotations

import contextlib
import os
import signal
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

DOWNLOAD_TRIES = 3
PROBE_SECONDS = 1200  # a first start (installs, downloads, GPU kernel compile) can take many minutes
WARM_BODY = b'{"state": "warm-up", "questions": {"q": {"type": "noul", "instructions": "Is this a test?"}}}'


def log(*args) -> None:
    print(time.strftime("%H:%M:%S"), *args, flush=True)


def _die_with_parent() -> None:
    """Kill the child if the gateway dies (Linux PR_SET_PDEATHSIG).

    Safe only because every launch comes from one long-lived thread: the signal fires when the thread that forked exits.
    """
    try:
        import ctypes

        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGTERM)
    except (OSError, AttributeError):
        pass


class Server:
    """A model served by a child process."""

    def __init__(self, entry: dict, cfg: dict):
        self.entry, self.cfg, self.port = entry, cfg, cfg["child_port"]
        self.proc: subprocess.Popen | None = None
        self.headers = {"Content-Type": "application/json"}

    def launch(self, cmd, env=None, cwd=None, ready="/healthz", ready_headers=None, wait_s=PROBE_SECONDS) -> None:
        """Start the process and block until `ready` answers, or raise with the end of its log."""
        self.log_path = Path(self.cfg["work"]) / f"{self.entry['id']}.log"
        with open(self.log_path, "w") as log_file:
            self.proc = subprocess.Popen(
                cmd,
                env={**os.environ, **(env or {})},
                cwd=cwd,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                start_new_session=True,
                preexec_fn=_die_with_parent,
            )
        for _ in range(wait_s):
            if self.proc.poll() is not None:
                raise RuntimeError(f"server exited ({self.proc.returncode}); end of {self.log_path}:\n{self._tail()}")
            if self._up(ready, ready_headers):
                return
            time.sleep(1)
        raise RuntimeError(f"server not healthy after {wait_s // 60} minutes; see {self.log_path}")

    def _tail(self, chars=600) -> str:
        return self.log_path.read_text()[-chars:]

    def _up(self, path, headers) -> bool:
        request = urllib.request.Request(f"http://127.0.0.1:{self.port}{path}", headers=headers or {})
        try:
            urllib.request.urlopen(request, timeout=2)
            return True
        except urllib.error.HTTPError as err:
            return err.code < 500  # up, just not happy with our probe (auth, method)
        except OSError:
            return False

    def answer(self, body: bytes) -> tuple[int, bytes]:
        """Forward one /v1/systemone request. Returns (status, body)."""
        request = urllib.request.Request(f"http://127.0.0.1:{self.port}/v1/systemone", body, self.headers)
        try:
            with urllib.request.urlopen(request, timeout=300) as response:
                return response.status, response.read()
        except urllib.error.HTTPError as err:
            return err.code, err.read()

    def warm(self) -> float:
        """One throwaway request so the first real call is not the cold one. Returns milliseconds; failures are ignored."""
        start = time.perf_counter()
        with contextlib.suppress(OSError):
            self.answer(WARM_BODY)
        return (time.perf_counter() - start) * 1000

    def stop(self) -> None:
        if self.proc and self.proc.poll() is None:
            try:
                os.killpg(self.proc.pid, signal.SIGTERM)
                self.proc.wait(30)
            except (OSError, subprocess.TimeoutExpired):
                with contextlib.suppress(OSError):
                    os.killpg(self.proc.pid, signal.SIGKILL)
        self.proc = None


def fetch(repo: str, name: str, dest: str) -> str:
    """A file from the Hugging Face Hub, or from a folder listed in CLEF_LOCAL (colon-separated) that already holds it."""
    for folder in filter(None, os.environ.get("CLEF_LOCAL", "").split(":")):
        if (Path(folder) / name).exists():
            return str(Path(folder) / name)
    from huggingface_hub import hf_hub_download

    for attempt in range(1, DOWNLOAD_TRIES + 1):  # a dropped connection resumes from the partial file
        try:
            return hf_hub_download(repo, name, local_dir=dest)
        except Exception as err:  # noqa: BLE001 - huggingface_hub raises several unrelated types for network trouble
            if attempt == DOWNLOAD_TRIES:
                raise
            log(f"download of {name} failed ({type(err).__name__}); retrying ({attempt}/{DOWNLOAD_TRIES - 1})")
            time.sleep(5 * attempt)


class Llama(Server):
    """llama.cpp's llama-server: native /v1/systemone, batches across slots, text and images."""

    def start(self) -> None:
        entry, cfg = self.entry, self.cfg
        model = fetch(entry["repo"], entry["file"], cfg["weights"])
        cmd = [
            cfg["llama_bin"], "-m", model, "--host", "127.0.0.1", "--port", str(self.port), "--alias", entry["id"],
            "-ngl", os.environ.get("CLEF_NGL", "999"), "-c", "65536", "-np", "4",
            # a decision is scored in one batch, so it must fit: the model's maximum input is 16384 tokens
            "-b", "16384", "-ub", "16384",
        ]  # fmt: skip
        if entry.get("mmproj"):
            cmd += ["--mmproj", fetch(entry["repo"], entry["mmproj"], cfg["weights"])]
        lib = str(Path(cfg["llama_bin"]).resolve().parent)
        env = {"LD_LIBRARY_PATH": f"{lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
        self.launch(cmd, env, ready="/health")


class Proc(Server):
    """Any model with a recipe: run its setup once, then its launch command."""

    def sub(self, text: str) -> str:
        return text.replace("${ROOT}", self.cfg["root"]).replace("${HERE}", self.cfg["here"]).replace("${port}", str(self.port))

    def start(self) -> None:
        entry, cfg = self.entry, self.cfg
        env = {
            "JEV_ROOT": cfg["root"],
            "HF_HOME": f"{cfg['root']}/data/models/hf-cache",
            "PATH": f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}",
            **{key: self.sub(value) for key, value in entry.get("env", {}).items()},
        }
        marker = Path(cfg["work"]) / f"{entry['id']}.setup-done"
        if entry.get("setup") and not marker.exists():
            self._setup(entry, env)
            marker.touch()
        headers = {key: self.sub(value) for key, value in entry.get("headers", {}).items()}
        self.headers.update(headers)
        cwd = self.sub(entry.get("cwd", cfg["root"]))
        self.launch([self.sub(part) for part in entry["cmd"]], env, cwd, entry.get("health", "/healthz"), headers)

    def _setup(self, entry: dict, env: dict) -> None:
        log(f"setup {entry['id']} (first time only: installs packages and downloads weights)")
        steps = ["command -v uv >/dev/null || pip install -q uv", *map(self.sub, entry["setup"])]
        setup_log = Path(self.cfg["work"]) / f"{entry['id']}.setup.log"
        with open(setup_log, "w") as out:
            for step in steps:
                result = subprocess.run(["bash", "-c", step], env={**os.environ, **env}, stdout=out, stderr=subprocess.STDOUT)
                if result.returncode:
                    raise RuntimeError(f"setup step failed ({result.returncode}): {step}; see {setup_log}")


KINDS = {"llama": Llama, "proc": Proc}

In [ ]:
%%writefile /content/gw/jevgw/manager.py
"""The gateway's state: which model is loaded, which are enabled, and per-model latency stats."""

from __future__ import annotations

import json
import statistics
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor

from . import catalog
from .backends import KINDS, Server, log
from .disk import Disk, DiskFull, files_of


class Refused(Exception):
    """A request that is valid but not allowed right now (disabled, does not fit); carries an HTTP status."""

    def __init__(self, message: str, status: int = 409):
        super().__init__(message)
        self.status = status


class Stats:
    """Request counts and recent latencies per model, for the panel and /v1/stats."""

    def __init__(self, keep: int = 500):
        self._lock = threading.Lock()
        self._n = defaultdict(int)
        self._errors = defaultdict(int)
        self._ms: dict[str, deque] = defaultdict(lambda: deque(maxlen=keep))

    def record(self, model: str | None, status: int, ms: float) -> None:
        with self._lock:
            self._n[model] += 1
            self._errors[model] += status >= 400
            self._ms[model].append(ms)

    def snapshot(self) -> dict:
        with self._lock:
            out = {}
            for model, n in self._n.items():
                ms = sorted(self._ms[model])
                out[str(model)] = {
                    "requests": n,
                    "errors": self._errors[model],
                    "p50_ms": round(statistics.median(ms), 1),
                    "p95_ms": round(ms[min(len(ms) - 1, int(len(ms) * 0.95))], 1),
                }
            return out


class Gateway:
    def __init__(self, models: dict[str, dict], cfg: dict, only: set[str] | None = None, disk: Disk | None = None):
        self.models, self.cfg, self.disk = models, cfg, disk
        self.disabled = {mid for mid in models if only is not None and mid not in only}
        self.gpu, self.gpu_mib = catalog.gpu_info()
        self.cur: Server | None = None
        self.cur_id: str | None = None
        self.loading: str | None = None
        self.load_s: float | None = None
        self.warm_ms: float | None = None
        self.error: str | None = None
        self.stats = Stats()
        self._future = None
        # One long-lived thread does every launch and stop (see backends._die_with_parent).
        self._pool = ThreadPoolExecutor(1, thread_name_prefix="launcher")

    def fits(self, entry: dict) -> bool:
        return catalog.fits(entry, self.gpu_mib)

    # -- state changes (each runs on the launcher thread, one at a time) ---------------------------------------------

    def select(self, model_id: str, wait: bool = True) -> None:
        """Unload the current model and load `model_id`. Safe to repeat: loaded or already loading is a no-op.

        With wait=True this blocks until the model answers. With wait=False it returns at once, the load runs in the background,
        and the state (loading, progress, error) is in status(). A failed load can simply be requested again.
        """
        entry = self._loadable(model_id)
        if self.cur_id == model_id and self.cur:
            return
        if self.loading and self.loading != model_id:
            raise Refused(f"still loading {self.loading}; wait for it to finish", 409)
        if self.loading != model_id:
            self.loading, self.error = model_id, None
            self._future = self._pool.submit(self._load, model_id, entry)
        if wait:
            self._future.result()

    def unload(self) -> None:
        self._pool.submit(self._unload).result()

    def set_enabled(self, model_id: str, enabled: bool) -> None:
        """Disabled models cannot be loaded. Disabling the loaded model unloads it."""
        if model_id not in self.models:
            raise KeyError(model_id)
        if enabled:
            self.disabled.discard(model_id)
            return
        self.disabled.add(model_id)
        if self.cur_id == model_id:
            self.unload()

    def purge(self, model_id: str) -> None:
        """Delete a model's downloaded files. The loaded model cannot be purged."""
        if model_id not in self.models:
            raise KeyError(model_id)
        if not self.disk:
            raise Refused("this gateway does not manage disk", 404)
        if model_id == self.cur_id or model_id == self.loading:
            raise Refused(f"{model_id} is loaded; unload it first")
        self.disk.delete(model_id, self.models)

    def _unload(self) -> None:
        if self.cur:
            log("unload", self.cur_id)
            self.cur.stop()
        self.cur, self.cur_id, self.load_s, self.warm_ms = None, None, None, None

    def _loadable(self, model_id: str) -> dict:
        """The catalog entry, or the reason it cannot be loaded right now."""
        if model_id not in self.models:
            raise KeyError(model_id)
        entry = self.models[model_id]
        if entry["kind"] not in KINDS:
            raise Refused(f"{model_id} has no recipe yet ({entry.get('status', 'kind ' + entry['kind'])})", 422)
        if model_id in self.disabled:
            raise Refused(f"{model_id} is disabled; enable it first", 403)
        if not self.fits(entry):
            raise Refused(f"{model_id} needs about {entry['vram_gib']} GiB; this GPU ({self.gpu or 'none'}) has "
                          f"{self.gpu_mib / 1024:.1f} GiB")  # fmt: skip
        return entry

    def _load(self, model_id: str, entry: dict) -> None:
        try:
            self._unload()
            self._make_room(entry)
            log("load", model_id)
            started = time.time()
            server = KINDS[entry["kind"]](entry, self.cfg)
            try:
                server.start()
            except Exception:
                server.stop()
                raise
            self.warm_ms = server.warm()
            if self.disk:
                self.disk.touch(model_id, time.time())
            self.cur, self.cur_id, self.load_s = server, model_id, round(time.time() - started, 1)
            log(f"ready {model_id} in {self.load_s}s (warm-up {self.warm_ms:.0f} ms)")
        except Exception as err:
            self.error = f"{type(err).__name__}: {err}"
            log("load failed:", self.error[:300])
            raise
        finally:
            self.loading = None

    def progress(self) -> dict | None:
        """What a load in flight is doing: downloading (with GiB done) or starting the model server."""
        if not self.loading:
            return None
        entry = self.models[self.loading]
        if self.disk and files_of(entry) and not self.disk.cached(entry):
            done, total = self.disk.downloaded_gib(entry)
            return {"phase": "downloading", "done_gib": round(done, 2), "total_gib": round(total, 2)}
        return {"phase": "starting"}

    def _make_room(self, entry: dict) -> None:
        if not self.disk or not files_of(entry):  # recipe models bring their own install
            return
        try:
            for gone in self.disk.make_room(entry, self.models):
                log(f"disk: deleted {gone} to make room for {entry['id']}")
        except DiskFull as err:
            raise Refused(str(err), 507) from None

    # -- serving -----------------------------------------------------------------------------------------------------

    def answer(self, body: bytes) -> tuple[int, bytes, str | None]:
        """Proxy one request to the loaded model. Returns (status, body, model id)."""
        server, model_id = self.cur, self.cur_id
        if not server:
            message = f"loading {self.loading}" if self.loading else "no model loaded"
            return 503, json.dumps({"error": message}).encode(), None
        start = time.perf_counter()
        try:
            status, out = server.answer(body)
        except OSError as err:
            status, out = 502, json.dumps({"error": f"model server unreachable: {err}"}).encode()
        self.stats.record(model_id, status, (time.perf_counter() - start) * 1000)
        return status, out, model_id

    def status(self) -> dict:
        keep = ("id", "name", "kind", "vram_gib", "vision", "gpu_hint", "note", "status", "verified")
        return {
            "gpu": self.gpu,
            "gpu_gib": round(self.gpu_mib / 1024, 1),
            "loaded": self.cur_id,
            "loading": self.loading,
            "progress": self.progress(),
            "load_s": self.load_s,
            "warm_ms": None if self.warm_ms is None else round(self.warm_ms),
            "error": self.error,
            "disk": self.disk.status(self.models) if self.disk else None,
            "models": [
                {**{k: e[k] for k in keep if k in e}, "fits": self.fits(e), "loaded": mid == self.cur_id,
                 "enabled": mid not in self.disabled, "runnable": catalog.runnable(e),
                 "cached": bool(self.disk and self.disk.cached(e)),
                 "disk_gib": round(e.get("file_gib", 0) + e.get("mmproj_gib", 0), 2) or None}
                for mid, e in self.models.items()
            ],
        }  # fmt: skip

    def shutdown(self) -> None:
        self._pool.submit(self._unload).result()
        self._pool.shutdown()

In [ ]:
%%writefile /content/gw/jevgw/tunnel.py
"""A free Cloudflare quick tunnel to the gateway, kept alive by a watchdog.

Quick tunnels are for testing: no uptime guarantee, 200 in-flight requests, no Server-Sent Events, and the URL changes every
time the tunnel restarts. The watchdog restarts a dead or disconnected tunnel and reports the new URL.

Health is read from cloudflared's own local /ready endpoint (live connections to Cloudflare's edge), never by fetching the public
hostname: a brand-new hostname can fail to resolve for minutes behind a caching resolver, and a probe that depends on it would
restart a healthy tunnel in a loop.
"""

from __future__ import annotations

import json
import platform
import re
import socket
import subprocess
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

from .backends import log

URL_PATTERN = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
RELEASE = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"


def _free_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def probe_ready(ready_url: str) -> bool:
    """True if cloudflared reports at least one live connection to the edge."""
    try:
        with urllib.request.urlopen(ready_url, timeout=5) as response:
            return json.load(response).get("readyConnections", 0) >= 1
    except (OSError, ValueError):
        return False


def launch_cloudflared(port: int, work: str, wait: int = 60):
    """Start cloudflared for http://127.0.0.1:port. Returns (process, https URL, local /ready URL).

    Raises RuntimeError if no URL appears. Returns once the tunnel reports a live connection, or after 20 s regardless.
    """
    exe = Path(work) / "cloudflared"
    if not exe.exists():
        arch = "arm64" if platform.machine() in ("aarch64", "arm64") else "amd64"
        urllib.request.urlretrieve(RELEASE.format(arch=arch), exe)
        exe.chmod(0o755)
    ready_url = f"http://127.0.0.1:{_free_port()}/ready"
    metrics = ready_url.removeprefix("http://").removesuffix("/ready")
    # http2: QUIC (UDP) is blocked on some networks, and the automatic fallback can be slow or stick.
    cmd = [
        str(exe),
        "tunnel",
        "--no-autoupdate",
        "--protocol",
        "http2",
        "--metrics",
        metrics,
        "--url",
        f"http://127.0.0.1:{port}",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    deadline, url = time.time() + wait, None
    for line in proc.stdout:  # the URL is printed on stderr, merged here
        found = URL_PATTERN.search(line)
        if found:
            url = found.group(0)
            break
        if time.time() > deadline or proc.poll() is not None:
            break
    if not url:
        proc.terminate()
        raise RuntimeError("no tunnel URL appeared; Cloudflare may be rate-limiting, try again in a minute")
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()  # keep draining its log
    for _ in range(20):
        if probe_ready(ready_url):
            break
        time.sleep(1)
    return proc, url, ready_url


class Tunnel:
    """Start, stop and watch a tunnel. `launcher` and `probe` can be replaced in tests."""

    def __init__(
        self, port: int, work: str, launcher=launch_cloudflared, probe=probe_ready, interval: float = 20.0, failures: int = 3
    ):
        self.port, self.work, self.launcher, self.probe = port, work, launcher, probe
        self.interval, self.max_failures = interval, failures
        self.proc = None
        self.url: str | None = None
        self.ready: str | None = None
        self.since: float | None = None
        self.restarts = 0
        self.error: str | None = None
        self._want = False
        self._lock = threading.RLock()
        self._wake = threading.Event()
        self._watchdog: threading.Thread | None = None

    def start(self) -> dict:
        with self._lock:
            self._want = True
            if not self._running():
                self._launch()
            if not self._watchdog or not self._watchdog.is_alive():
                self._watchdog = threading.Thread(target=self._watch, daemon=True, name="tunnel-watchdog")
                self._watchdog.start()
            return self.status()

    def stop(self) -> dict:
        self._want = False  # before taking the lock, so a watchdog that is backing off wakes up and lets go
        self._wake.set()
        with self._lock:
            self._kill()
            self.url = self.since = None
            return self.status()

    def status(self) -> dict:
        return {"running": self._running(), "url": self.url, "restarts": self.restarts, "error": self.error,
                "up_s": None if self.since is None else round(time.time() - self.since)}  # fmt: skip

    def _running(self) -> bool:
        return bool(self.proc) and self.proc.poll() is None

    def _launch(self) -> None:
        self.proc, self.url, self.ready = self.launcher(self.port, self.work)
        self.since, self.error = time.time(), None
        log("tunnel up", self.url)

    def _kill(self) -> None:
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(10)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        self.proc = None

    def _watch(self) -> None:
        bad, delay = 0, 5.0
        while self._want:
            self._wake.wait(self.interval)
            self._wake.clear()
            backoff = 0.0
            with self._lock:
                if not self._want:
                    return
                bad = 0 if self._running() and self.probe(self.ready) else bad + 1
                if self._running() and bad < self.max_failures:
                    continue  # healthy, or one probe failure not yet repeated enough to act on
                log("tunnel down; restarting" if not self._running() else "tunnel disconnected from the edge; restarting")
                self._kill()
                try:
                    self._launch()
                    self.restarts, bad, delay = self.restarts + 1, 0, 5.0
                except RuntimeError as err:
                    self.error, backoff, delay = str(err), delay, min(delay * 2, 120.0)
                    log("tunnel restart failed:", err)
            if backoff:
                self._wake.wait(backoff)  # outside the lock: stop() must never wait on this

In [ ]:
%%writefile /content/gw/jevgw/server.py
"""The HTTP API and the control panel.

Public:   GET /healthz.  GET / (the panel) answers only requests that did not come through the tunnel.
Key:      GET  /v1/models /v1/stats /v1/tunnel
          POST /v1/systemone  /admin/select  /admin/unload  /admin/enable  /admin/purge  /admin/tunnel
The key is "Authorization: Bearer <key>" or "X-API-Key: <key>". Every response carries X-Request-Id and X-Gateway-Version.
"""

from __future__ import annotations

import json
import secrets
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

from . import __version__
from .manager import Gateway, Refused
from .tunnel import Tunnel

HERE = Path(__file__).resolve().parent
MAX_BODY = 32 * 1024 * 1024  # a few base64 images
PUBLIC = {("GET", "/"), ("GET", "/healthz")}
TUNNEL_HEADERS = ("Cf-Ray", "Cf-Connecting-Ip")  # Cloudflare adds these to everything it forwards


class RateLimit:
    """A token bucket: `per_minute` requests a minute, bursting up to `burst`. per_minute=0 turns it off."""

    def __init__(self, per_minute: int, burst: int | None = None):
        self.rate, self.burst = per_minute / 60.0, float(burst or max(per_minute // 6, 1))
        self.tokens, self.last, self._lock = self.burst, time.monotonic(), threading.Lock()

    def take(self) -> float:
        """0 if allowed, else seconds to wait."""
        if not self.rate:
            return 0.0
        with self._lock:
            now = time.monotonic()
            self.tokens = min(self.burst, self.tokens + (now - self.last) * self.rate)
            self.last = now
            if self.tokens >= 1:
                self.tokens -= 1
                return 0.0
            return (1 - self.tokens) / self.rate


class App:
    """Everything a request handler needs."""

    def __init__(self, gateway: Gateway, key: str, tunnel: Tunnel | None = None, work: str | None = None,
                 max_inflight: int = 64, per_minute: int = 600):  # fmt: skip
        self.gw, self.key, self.tunnel = gateway, key, tunnel
        self.started = time.time()
        self.inflight = threading.BoundedSemaphore(max_inflight)
        self.limit = RateLimit(per_minute)
        self.panel = (HERE / "ui" / "index.html").read_bytes()
        self.access_log = Path(work) / "requests.jsonl" if work else None
        self._log_lock = threading.Lock()

    def record(self, entry: dict) -> None:
        """One JSON line per request: no bodies, so nothing a caller sent is stored."""
        if self.access_log:
            with self._log_lock, open(self.access_log, "a") as out:
                out.write(json.dumps(entry) + "\n")


def make_handler(app: App):
    class Handler(BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        # -- plumbing ------------------------------------------------------------------------------------------------

        def _send(self, status: int, body, extra: dict | None = None, ctype: str = "application/json") -> None:
            raw = body if isinstance(body, bytes) else json.dumps(body).encode()
            self.send_response(status)
            headers = {
                "Content-Type": ctype,
                "Content-Length": str(len(raw)),
                "X-Request-Id": self.rid,
                "X-Gateway-Version": __version__,
                "X-Model": str(app.gw.cur_id),
                "Access-Control-Allow-Origin": "*",
                "Access-Control-Allow-Headers": "Authorization, Content-Type, X-API-Key, X-Request-Id",
                "Access-Control-Expose-Headers": "X-Latency-Ms, X-Model, X-Request-Id",
                **(extra or {}),
            }
            if self.command == "POST" and not self.body_read:
                headers["Connection"] = "close"  # an unread body would corrupt the next request on this connection
                self.close_connection = True
            for name, value in headers.items():
                self.send_header(name, value)
            self.end_headers()
            self.wfile.write(raw)
            self.status = status

        def _authed(self) -> bool:
            token = self.headers.get("Authorization", "").removeprefix("Bearer ").strip() or self.headers.get("X-API-Key", "")
            return secrets.compare_digest(token.encode(), app.key.encode())

        def _via_tunnel(self) -> bool:
            return any(self.headers.get(name) for name in TUNNEL_HEADERS)

        def _read(self) -> bytes | None:
            length = int(self.headers.get("Content-Length") or 0)
            if length > MAX_BODY:
                self._send(413, {"error": f"body over {MAX_BODY // 2**20} MB"})
                return None
            self.body_read = True
            return self.rfile.read(length)

        def _json(self, body: bytes) -> dict:
            try:
                data = json.loads(body or b"{}")
            except ValueError:
                raise Refused("body is not valid JSON", 400) from None
            if not isinstance(data, dict):
                raise Refused("body must be a JSON object", 400)
            return data

        def _dispatch(self, method: str) -> None:
            self.rid = (self.headers.get("X-Request-Id") or "")[:64] or uuid.uuid4().hex[:12]
            self.status, self.model, self.body_read, started = 0, None, False, time.perf_counter()
            path = self.path.split("?")[0]
            try:
                if method == "OPTIONS":
                    return self._send(204, b"", {"Access-Control-Allow-Methods": "GET, POST, OPTIONS"})
                route = ROUTES.get((method, path))
                if route is None or (path == "/" and self._via_tunnel()):
                    return self._send(404, {"error": "not found"})  # the panel is for the notebook, not the public URL
                if (method, path) not in PUBLIC and not self._authed():
                    return self._send(401, {"error": "missing or wrong API key (Authorization: Bearer <key>)"})
                body = self._read() if method == "POST" else b""
                if body is None:
                    return None
                getattr(self, route)(body)
            except Refused as err:
                self._send(err.status, {"error": str(err)})
            except KeyError as err:
                self._send(404, {"error": f"unknown model {err}", "models": list(app.gw.models)})
            except Exception as err:  # a failed load or a bug: tell the caller, keep serving
                self._send(500, {"error": f"{type(err).__name__}: {err}"[:1500]})
            finally:
                app.record({"ts": round(time.time(), 3), "id": self.rid, "method": method, "path": path, "status": self.status,
                            "ms": round((time.perf_counter() - started) * 1000, 1), "model": self.model,
                            "ip": self.headers.get("CF-Connecting-IP") or self.client_address[0]})  # fmt: skip

        def do_GET(self):
            self._dispatch("GET")

        def do_POST(self):
            self._dispatch("POST")

        def do_OPTIONS(self):
            self._dispatch("OPTIONS")

        def log_message(self, *args):
            pass

        # -- routes --------------------------------------------------------------------------------------------------

        def panel(self, _body):
            self._send(200, app.panel, ctype="text/html; charset=utf-8")

        def health(self, _body):
            gw = app.gw
            self._send(200 if gw.cur else 503, {"ok": bool(gw.cur), "loaded": gw.cur_id, "loading": gw.loading})

        def models(self, _body):
            self._send(200, app.gw.status())

        def stats(self, _body):
            self._send(200, {"version": __version__, "up_s": round(time.time() - app.started), "stats": app.gw.stats.snapshot()})

        def tunnel_status(self, _body):
            self._send(200, app.tunnel.status() if app.tunnel else {"running": False, "available": False})

        def systemone(self, body):
            wait = app.limit.take()
            if wait:
                return self._send(429, {"error": "rate limit"}, {"Retry-After": str(max(1, round(wait)))})
            if not app.inflight.acquire(blocking=False):
                return self._send(429, {"error": "too many requests in flight"}, {"Retry-After": "1"})
            try:
                started = time.perf_counter()
                status, out, self.model = app.gw.answer(body)
                extra = (
                    {"Retry-After": "10"} if status == 503 else {"X-Latency-Ms": f"{(time.perf_counter() - started) * 1000:.1f}"}
                )
                self._send(status, out, extra)
            finally:
                app.inflight.release()

        def select(self, body):
            data = self._json(body)
            wait = bool(data.get("wait", True))
            app.gw.select(str(data.get("model", "")), wait)
            self._send(200 if wait else 202, app.gw.status())

        def unload(self, _body):
            app.gw.unload()
            self._send(200, app.gw.status())

        def enable(self, body):
            data = self._json(body)
            app.gw.set_enabled(str(data.get("model", "")), bool(data.get("enabled", True)))
            self._send(200, app.gw.status())

        def purge(self, body):
            app.gw.purge(str(self._json(body).get("model", "")))
            self._send(200, app.gw.status())

        def tunnel_action(self, body):
            if not app.tunnel:
                raise Refused("this gateway was started without tunnel support", 404)
            action = self._json(body).get("action")
            if action not in ("start", "stop"):
                raise Refused('action must be "start" or "stop"', 400)
            try:
                self._send(200, app.tunnel.start() if action == "start" else app.tunnel.stop())
            except RuntimeError as err:
                raise Refused(str(err), 502) from None

    ROUTES = {
        ("GET", "/"): "panel", ("GET", "/healthz"): "health", ("GET", "/v1/models"): "models", ("GET", "/v1/stats"): "stats",
        ("GET", "/v1/tunnel"): "tunnel_status", ("POST", "/v1/systemone"): "systemone", ("POST", "/admin/select"): "select",
        ("POST", "/admin/unload"): "unload", ("POST", "/admin/enable"): "enable", ("POST", "/admin/purge"): "purge",
        ("POST", "/admin/tunnel"): "tunnel_action",
    }  # fmt: skip
    return Handler


def serve(app: App, port: int, host: str = "127.0.0.1") -> ThreadingHTTPServer:
    """Start the server on a background thread and return it (call .shutdown() to stop)."""
    server = ThreadingHTTPServer((host, port), make_handler(app))
    server.daemon_threads = True
    threading.Thread(target=server.serve_forever, daemon=True, name="http").start()
    return server

In [ ]:
%%writefile /content/gw/jevgw/client.py
"""A client for the gateway: text, image and video calls with response times.

    python -m jevgw.client --url http://127.0.0.1:8000 --key KEY --model clef-flash-q4km

Video goes in as sampled frames: /v1/systemone takes `images`, not a video file. Standard library, plus Pillow and ffmpeg
(both on Colab) for the built-in image and video samples.
"""

from __future__ import annotations

import argparse
import base64
import io
import json
import statistics
import subprocess
import tempfile
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path

TEXT = {
    "state": "Our checkout started returning errors and orders are blocked.",
    "questions": {
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {"billing": "Payments or invoices", "technical": "Bugs or outages"},
        },
        "urgency": {"type": "score", "instructions": "How urgent is it?", "criteria": ["Can wait", "This week", "Today"]},
        "outage": {"type": "noul", "instructions": "Is a service down?"},
    },
}


@dataclass
class Response:
    status: int
    data: dict
    headers: dict
    ms: float  # round trip, client side

    @property
    def model_ms(self) -> float:
        """Time inside the model server, as the gateway reports it (nan if it did not)."""
        return float(self.headers.get("X-Latency-Ms", "nan"))


class Client:
    def __init__(self, url: str = "http://127.0.0.1:8000", key: str = ""):
        self.url, self.key = url.rstrip("/"), key

    def call(self, method: str, path: str, body: dict | None = None, timeout: float = 300) -> Response:
        data = None if body is None else json.dumps(body).encode()
        request = urllib.request.Request(
            self.url + path, data, {"Content-Type": "application/json", "Authorization": f"Bearer {self.key}"}, method=method
        )
        start = time.perf_counter()
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                status, raw, headers = response.status, response.read(), dict(response.headers)
        except urllib.error.HTTPError as err:
            status, raw, headers = err.code, err.read(), dict(err.headers)
        return Response(status, json.loads(raw or b"{}"), headers, (time.perf_counter() - start) * 1000)

    # -- one method per endpoint ---------------------------------------------------------------------------------------

    def systemone(self, body: dict) -> Response:
        return self.call("POST", "/v1/systemone", body)

    def models(self) -> dict:
        return self.call("GET", "/v1/models").data

    def stats(self) -> dict:
        return self.call("GET", "/v1/stats").data["stats"]

    def select(self, model: str, wait: bool = True) -> dict:
        """Load a model (the current one is unloaded first). Raises if refused.

        wait=True blocks until it answers. wait=False returns at once and the load runs in the background: watch `models()["progress"]`,
        or use `jevgw.notebook.wait_ready`. Asking again for a model that is loading or loaded does nothing.
        """
        response = self.call("POST", "/admin/select", {"model": model, "wait": wait}, timeout=3600)
        if response.status not in (200, 202):
            raise RuntimeError(f"{model}: HTTP {response.status}: {response.data.get('error')}")
        return response.data

    def unload(self) -> dict:
        return self.call("POST", "/admin/unload", {}).data

    def enable(self, model: str, enabled: bool = True) -> dict:
        return self.call("POST", "/admin/enable", {"model": model, "enabled": enabled}).data

    def tunnel(self, action: str | None = None) -> dict:
        """The tunnel's status, or "start" / "stop" it. Raises if it cannot start."""
        if action is None:
            return self.call("GET", "/v1/tunnel").data
        response = self.call("POST", "/admin/tunnel", {"action": action}, timeout=120)
        if response.status != 200:
            raise RuntimeError(f"tunnel {action}: HTTP {response.status}: {response.data.get('error')}")
        return response.data


# -- samples and timing --------------------------------------------------------------------------------------------------


def data_url(image, fmt: str = "JPEG") -> str:
    """A PIL image, or a path to one, as a data URL."""
    from PIL import Image

    img = image if hasattr(image, "save") else Image.open(image)
    buffer = io.BytesIO()
    img.convert("RGB").save(buffer, fmt, quality=90)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buffer.getvalue()).decode()


def sample_image():
    """An invoice-like picture with a known total, drawn here, so no third-party image is needed."""
    from PIL import Image, ImageDraw

    img = Image.new("RGB", (640, 400), "white")
    draw = ImageDraw.Draw(img)
    lines = ["ACME SUPPLIES", "Invoice 2024-0117", "Paper A4 x2       8.40", "Stapler           3.50", "TOTAL EUR       11.90"]
    for i, line in enumerate(lines):
        draw.text((40, 40 + i * 50), line, fill="black")
    return img


def video_frames(path, count: int = 8) -> list[str]:
    """`count` evenly spaced frames of a video, as data URLs."""
    from PIL import Image

    duration = float(
        subprocess.check_output(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)]
        ).strip()
    )
    frames = []
    for i in range(count):
        at = duration * (i + 0.5) / count
        png = subprocess.check_output(
            [
                "ffmpeg",
                "-loglevel",
                "error",
                "-ss",
                f"{at:.3f}",
                "-i",
                str(path),
                "-frames:v",
                "1",
                "-f",
                "image2pipe",
                "-vcodec",
                "png",
                "-",
            ]
        )
        frames.append(data_url(Image.open(io.BytesIO(png))))
    return frames


def sample_video_frames(count: int = 6) -> list[str]:
    """A 4-second clip of a red square crossing the frame, made with ffmpeg, sampled back into frames."""
    from PIL import Image, ImageDraw

    folder = Path(tempfile.mkdtemp())
    for i in range(40):
        frame = Image.new("RGB", (320, 180), "white")
        ImageDraw.Draw(frame).rectangle([10 + i * 7, 60, 50 + i * 7, 100], fill="red")
        frame.save(folder / f"f{i:03d}.png")
    clip = folder / "clip.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-loglevel",
            "error",
            "-y",
            "-framerate",
            "10",
            "-i",
            str(folder / "f%03d.png"),
            "-pix_fmt",
            "yuv420p",
            str(clip),
        ],
        check=True,
    )
    return video_frames(clip, count)


def bench(client: Client, label: str, body: dict, n: int = 5) -> dict | None:
    """One warm-up call, then n timed calls. Prints and returns a row, or None if the call failed."""
    first = client.systemone(body)
    if first.status != 200:
        print(f"{label:<28} HTTP {first.status}: {first.data.get('error', first.data)}")
        return None
    runs = [client.systemone(body) for _ in range(n)]
    round_trip = [r.ms for r in runs]
    row = {
        "test": label,
        "n": n,
        "round_trip_p50_ms": round(statistics.median(round_trip)),
        "round_trip_max_ms": round(max(round_trip)),
        "model_p50_ms": round(statistics.median(r.model_ms for r in runs)),
        "answers": first.data.get("answers", first.data),
    }
    print(
        f"{label:<28} p50 {row['round_trip_p50_ms']:>5} ms (max {row['round_trip_max_ms']:>5}), model {row['model_p50_ms']:>5} ms"
    )
    return row


def demo(client: Client, image=None, video=None, frames: int = 6, n: int = 5) -> list[dict]:
    """Text, image and video tests against whatever model is loaded."""
    rows = [bench(client, "text, 3 questions", TEXT, n)]
    invoice = {
        "total": {
            "type": "choice",
            "instructions": "What is the total on this invoice?",
            "criteria": {"11.90": None, "61.90": None, "8.40": None},
        }
    }
    rows.append(
        bench(
            client,
            "image, 1 question",
            {"state": "An invoice.", "images": [data_url(image or sample_image())], "questions": invoice},
            n,
        )
    )
    shots = video_frames(video, frames) if video else sample_video_frames(frames)
    moving = {"moving": {"type": "noul", "instructions": "Is a red square visible in these frames?"}}
    rows.append(
        bench(
            client,
            f"video as {len(shots)} frames",
            {"state": "Frames sampled evenly from one video.", "images": shots, "questions": moving},
            n,
        )
    )
    return [row for row in rows if row]


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--url", default="http://127.0.0.1:8000")
    ap.add_argument("--key", required=True)
    ap.add_argument("--model", help="load this model first")
    ap.add_argument("--image")
    ap.add_argument("--video")
    ap.add_argument("--frames", type=int, default=6)
    ap.add_argument("-n", type=int, default=5, help="timed calls per test")
    args = ap.parse_args()
    client = Client(args.url, args.key)
    if args.model:
        start = time.time()
        client.select(args.model)
        print(f"loaded {args.model} in {time.time() - start:.0f}s")
    for row in demo(client, args.image, args.video, args.frames, args.n):
        print(json.dumps(row["answers"])[:300])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/gw/jevgw/notebook.py
"""Helpers for the Colab notebook. Every function is safe to call again: re-running a cell never needs an "undo" first.

ensure_llama()   install the llama.cpp server, or notice it is already installed and working
start()          start the gateway, or reuse the one already running (same port, same key)
wait_ready()     load a model in the background, show download progress, retry a failed load
stop()           stop the gateway (and with it the model server and the tunnel)
reset()          stop everything and clear the half-finished state, optionally deleting downloaded models
"""

from __future__ import annotations

import contextlib
import json
import os
import secrets
import shutil
import signal
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

from .client import Client

APP = Path(__file__).resolve().parent.parent
WORK = Path(os.environ.get("JEVGW_WORK", "/content/work"))  # logs, state and downloaded models
LLAMA_DIR = Path(os.environ.get("JEVGW_LLAMA", "/content/llama"))


def _say(*parts) -> None:
    print(*parts, flush=True)


# -- llama.cpp ---------------------------------------------------------------------------------------------------------


def _llama_works(binary: Path) -> bool:
    try:
        env = {**os.environ, "LD_LIBRARY_PATH": f"{binary.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
        return subprocess.run([str(binary), "--version"], env=env, capture_output=True, timeout=60).returncode == 0
    except (OSError, subprocess.SubprocessError):
        return False


def ensure_llama(t4_url: str = "") -> str:
    """The path to a working llama-server. A missing, half-downloaded or broken install is removed and done again."""
    binary = LLAMA_DIR / "bin" / "llama-server"
    if binary.exists() and _llama_works(binary):
        _say("llama.cpp is already installed:", binary)
        return str(binary)
    if LLAMA_DIR.exists():
        _say("removing a broken llama.cpp install and starting it again")
        shutil.rmtree(LLAMA_DIR)
    _say("installing llama.cpp (about a minute)")
    result = subprocess.run(
        ["bash", str(APP / "setup_llama.sh")],
        capture_output=True,
        text=True,
        env={**os.environ, "LLAMA_DIR": str(LLAMA_DIR), "LLAMA_T4_URL": t4_url},
    )
    if result.stderr.strip():
        _say(result.stderr.strip())
    if result.returncode:
        raise RuntimeError("llama.cpp install failed; run this cell again to retry.\n" + (result.stdout + result.stderr)[-1500:])
    return result.stdout.strip().splitlines()[-1]


# -- the gateway process -----------------------------------------------------------------------------------------------


def _state_file() -> Path:
    return WORK / "gateway.json"


def _read_state() -> dict | None:
    try:
        return json.loads(_state_file().read_text())
    except (OSError, ValueError):
        return None


def _alive(pid: int) -> bool:
    """True if the process is running. A child of this kernel that has exited is reaped here, so it never looks alive."""
    try:
        if os.waitpid(pid, os.WNOHANG)[0]:
            return False
    except ChildProcessError:
        pass  # not our child: fall through to the plain check
    except OSError:
        return False
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    return True


def _is_ours(port: int) -> bool:
    """True if a jevgw gateway answers on the port (a 503 with its JSON body counts: it is up, just no model yet)."""
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/healthz", timeout=3) as response:
            body = response.read()
    except urllib.error.HTTPError as err:
        body = err.read()
    except OSError:
        return False
    try:
        return "loaded" in json.loads(body)
    except ValueError:
        return False


def _free_port(preferred: int) -> int:
    for port in range(preferred, preferred + 50):
        with socket.socket() as sock:
            try:
                sock.bind(("127.0.0.1", port))
            except OSError:
                continue
            return port
    raise RuntimeError(f"no free port from {preferred} to {preferred + 49}")


def start(llama_bin: str, keep: int = 2, port: int = 8000, key: str = "") -> tuple[Client, dict]:
    """Start the gateway, or reuse the running one. Returns (client, {"port", "key", "reused"}). The key is printed.

    `key` is the API key to use; leave it empty to get a random one. A running gateway with a different key is replaced.
    A gateway left by an earlier run of this cell is reused if it still answers, and stopped if it does not. A busy port is skipped.
    """
    state = _read_state()
    if state and _alive(state["pid"]) and _is_ours(state["port"]) and key in ("", state["key"]):
        _say(f"the gateway is already running on port {state['port']}; reusing it")
        _say(f"API key: {state['key']}")
        return Client(f"http://127.0.0.1:{state['port']}", state["key"]), {**state, "reused": True}
    stop()
    WORK.mkdir(parents=True, exist_ok=True)
    port, key = _free_port(port), key or secrets.token_urlsafe(12)
    log_path = WORK / "gateway.log"
    with open(log_path, "w") as log:
        proc = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "jevgw",
                "--port",
                str(port),
                "--key",
                key,
                "--llama-bin",
                llama_bin,
                "--work",
                str(WORK),
                "--keep",
                str(keep),
            ],
            cwd=APP,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    for _ in range(60):
        if proc.poll() is not None:
            raise RuntimeError(f"the gateway exited ({proc.returncode}); run this cell again.\n{log_path.read_text()[-1500:]}")
        if _is_ours(port):
            break
        time.sleep(1)
    else:
        proc.terminate()
        raise RuntimeError(f"the gateway did not answer within a minute; see {log_path}")
    state = {"pid": proc.pid, "port": port, "key": key}
    _state_file().write_text(json.dumps(state))
    _say(f"the gateway is running on port {port}")
    _say(f"API key: {key}")
    return Client(f"http://127.0.0.1:{port}", key), {**state, "reused": False}


def show_panel(info: dict) -> None:
    """The control panel inside the notebook (on Colab), or its address anywhere else."""
    try:
        from google.colab import output

        output.serve_kernel_port_as_iframe(info["port"], path=f"/#key={info['key']}", height="900")
    except ImportError:
        _say(f"panel: http://127.0.0.1:{info['port']}/#key={info['key']}")


def stop() -> None:
    """Stop the gateway if one is running. The model server and the tunnel stop with it. Does nothing if none is."""
    state = _read_state()
    if state and _alive(state["pid"]):
        with contextlib.suppress(OSError):
            os.kill(state["pid"], signal.SIGTERM)
        for _ in range(40):
            if not _alive(state["pid"]):
                break
            time.sleep(0.5)
        else:
            with contextlib.suppress(OSError):
                os.kill(state["pid"], signal.SIGKILL)
    _state_file().unlink(missing_ok=True)


def reset(delete_models: bool = False) -> None:
    """Stop everything and clear the half-finished state, so the next cells start clean. Downloaded models stay unless asked."""
    stop()
    for name in ("gateway.json", "gateway.log", "requests.jsonl", "disk.json"):
        (WORK / name).unlink(missing_ok=True)
    if delete_models:
        shutil.rmtree(WORK / "weights", ignore_errors=True)
    _say("reset done" + (" (downloaded models deleted)" if delete_models else " (downloaded models kept)"))


# -- loading a model ---------------------------------------------------------------------------------------------------


def _describe(status: dict) -> str:
    progress = status.get("progress") or {}
    if progress.get("phase") == "downloading":
        total, done = progress["total_gib"], progress["done_gib"]
        return f"downloading {done:.1f} of {total:.1f} GiB ({done / total:.0%})" if total else "downloading"
    return "starting the model server (the first start on a T4 can take several minutes)"


def wait_ready(client: Client, model: str, timeout: float = 3600, poll: float = 5, retries: int = 2) -> dict:
    """Load `model` in the background and wait for it, printing progress. Run it again at any point: it picks up where things are.

    A load that fails (a dropped download, a crashed server) is retried up to `retries` times; the last error is raised.
    Refusals (does not fit this GPU, not enough disk, disabled) raise at once with the reason.
    """
    deadline, failures, last_line = time.time() + timeout, 0, ""
    client.select(model, wait=False)
    while time.time() < deadline:
        status = client.models()
        if status["loaded"] == model:
            _say(
                f"\n{model} is loaded (load {status['load_s']} s, warm-up {status['warm_ms']} ms). Free disk {status['disk']['free_gib']} GiB."
            )
            return status
        if status["loading"] == model:
            line = f"  {model}: {_describe(status)}"
            if line != last_line:
                _say(line)
                last_line = line
        else:  # not loading and not loaded: it failed or was unloaded
            failures += 1
            if failures > retries:
                raise RuntimeError(f"{model} failed to load {failures} times. Last error: {status.get('error')}")
            _say(f"  load failed ({status.get('error')}); retrying ({failures}/{retries})")
            client.select(model, wait=False)
        time.sleep(poll)
    raise TimeoutError(f"{model} was not ready after {timeout / 60:.0f} minutes; run this cell again to keep waiting")

In [ ]:
%%writefile /content/gw/jevgw/__main__.py
"""python -m jevgw [options]: start the gateway. Prints one JSON line with the local URL and the API key."""

from __future__ import annotations

import argparse
import json
import os
import secrets
import signal
import time
from pathlib import Path

from . import __version__, catalog
from .disk import Disk
from .manager import Gateway
from .server import App, serve
from .tunnel import Tunnel

ROOT = Path(__file__).resolve().parent.parent


def parse(argv=None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(prog="jevgw", description=__doc__)
    ap.add_argument("--catalog", default=str(ROOT / "models.json"), help="the model list and recipes")
    ap.add_argument("--port", type=int, default=8000)
    ap.add_argument("--host", default="127.0.0.1", help="leave as is: the tunnel and the Colab panel reach it locally")
    ap.add_argument(
        "--key", default=os.environ.get("GATEWAY_KEY") or secrets.token_urlsafe(18), help="API key (default: random, printed)"
    )
    ap.add_argument("--model", help="load this model at start")
    ap.add_argument("--only", help="comma-separated ids: every other model starts disabled")
    ap.add_argument("--root", default=str(ROOT / "vendor"), help="where models/ and data/ are installed (the JEV_ROOT layout)")
    ap.add_argument("--work", default=str(ROOT / "work"), help="logs, downloaded GGUF files, setup markers")
    ap.add_argument("--keep", type=int, default=2, help="downloaded models kept on disk; the least recently used are deleted")
    ap.add_argument("--disk-margin-gib", type=float, default=5.0, help="free disk space always left alone")
    ap.add_argument("--llama-bin", default=os.environ.get("LLAMA_BIN", "llama-server"))
    ap.add_argument(
        "--tunnel", action="store_true", help="start a Cloudflare quick tunnel at launch (the panel can also start one)"
    )
    ap.add_argument("--no-tunnel-support", action="store_true", help="refuse tunnel requests entirely")
    ap.add_argument("--max-inflight", type=int, default=64, help="concurrent /v1/systemone calls before answering 429")
    ap.add_argument("--per-minute", type=int, default=600, help="rate limit for /v1/systemone; 0 turns it off")
    return ap.parse_args(argv)


def main(argv=None) -> None:
    args = parse(argv)
    work = Path(args.work)
    (work / "weights").mkdir(parents=True, exist_ok=True)
    cfg = {"llama_bin": args.llama_bin, "weights": str(work / "weights"), "work": str(work), "root": args.root,
           "here": str(ROOT), "child_port": args.port + 1}  # fmt: skip
    models = catalog.load(args.catalog)
    only = set(args.only.split(",")) if args.only else None
    disk = Disk(cfg["weights"], work / "disk.json", args.keep, args.disk_margin_gib)
    gateway = Gateway(models, cfg, only, disk)
    tunnel = None if args.no_tunnel_support else Tunnel(args.port, str(work))
    app = App(gateway, args.key, tunnel, str(work), args.max_inflight, args.per_minute)
    server = serve(app, args.port, args.host)
    signal.signal(signal.SIGTERM, lambda *_: (_ for _ in ()).throw(KeyboardInterrupt()))
    info = {
        "version": __version__,
        "local": f"http://{args.host}:{args.port}",
        "key": args.key,
        "gpu": gateway.gpu,
        "gpu_gib": round(gateway.gpu_mib / 1024, 1),
    }
    try:
        if args.tunnel and tunnel:
            info["url"] = tunnel.start()["url"]
        print("GATEWAY " + json.dumps(info), flush=True)
        if args.model:
            gateway.select(args.model)
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        pass
    finally:
        if tunnel:
            tunnel.stop()
        server.shutdown()
        gateway.shutdown()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/gw/jevgw/ui/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Jev gateway</title>
<style>
  :root {
    --bg: #fafafa; --surface: #fff; --surface-2: #f4f4f5; --border: #e4e4e7; --text: #18181b; --text-2: #52525b; --text-3: #71717a;
    --accent: #1d4ed8; --good: #15803d; --warn: #a16207; --bad: #b91c1c; --radius: 6px;
    --mono: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
  }
  @media (prefers-color-scheme: dark) {
    :root { --bg: #0b0b0d; --surface: #131316; --surface-2: #1b1b1f; --border: #2a2a30; --text: #ececf0; --text-2: #a1a1aa; --text-3: #7c7c86;
            --accent: #6d9cff; --good: #4ade80; --warn: #facc15; --bad: #f87171; }
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--text); font: 14px/1.45 system-ui, -apple-system, "Segoe UI", sans-serif; }
  main { max-width: 1100px; margin: 0 auto; padding: 16px; display: grid; gap: 16px; }
  h1 { font-size: 18px; margin: 0; } h2 { font-size: 14px; margin: 0 0 8px; }
  header { display: flex; flex-wrap: wrap; gap: 8px 16px; align-items: center; justify-content: space-between; }
  .card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 12px 14px; }
  .row { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
  .muted { color: var(--text-2); } .small { font-size: 12px; } .mono { font-family: var(--mono); font-size: 12px; }
  .chip { display: inline-block; padding: 1px 8px; border-radius: 999px; border: 1px solid var(--border); font-size: 12px; background: var(--surface-2); }
  .chip.good { color: var(--good); border-color: currentColor; } .chip.warn { color: var(--warn); border-color: currentColor; }
  .chip.bad { color: var(--bad); border-color: currentColor; }
  button { font: inherit; color: var(--text); background: var(--surface-2); border: 1px solid var(--border); border-radius: var(--radius); padding: 4px 10px; cursor: pointer; }
  button:hover:not(:disabled) { border-color: var(--accent); } button:disabled { opacity: .45; cursor: not-allowed; }
  button.primary { background: var(--accent); color: #fff; border-color: var(--accent); }
  button:focus-visible, input:focus-visible, textarea:focus-visible, summary:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
  input, textarea, select { font: inherit; color: var(--text); background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 4px 8px; }
  textarea { width: 100%; font-family: var(--mono); font-size: 12px; }
  table { width: 100%; border-collapse: collapse; } th, td { text-align: left; padding: 6px 8px; border-bottom: 1px solid var(--border); vertical-align: top; }
  th { font-size: 11px; text-transform: uppercase; letter-spacing: .05em; color: var(--text-3); font-weight: 600; }
  td.num, th.num { text-align: right; font-variant-numeric: tabular-nums; }
  tr.loaded td { background: color-mix(in srgb, var(--good) 8%, var(--surface)); }
  pre { margin: 8px 0 0; padding: 8px 10px; background: var(--surface-2); border: 1px solid var(--border); border-radius: var(--radius); overflow: auto; max-height: 280px; font: 12px var(--mono); white-space: pre-wrap; overflow-wrap: anywhere; }
  .tabs { display: flex; gap: 4px; margin-bottom: 8px; } .tabs button[aria-selected="true"] { border-color: var(--accent); color: var(--accent); }
  .grid2 { display: grid; grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); gap: 16px; }
  .table-wrap { overflow-x: auto; }
  [hidden] { display: none !important; }
</style>
</head>
<body>
<main>
  <header>
    <div class="row"><h1>Jev gateway</h1><span class="chip" id="version"></span><span class="chip" id="gpu">GPU: ?</span><span class="chip" id="disk">disk: ?</span></div>
    <div class="row" role="status" aria-live="polite"><span class="chip" id="state">not connected</span><span class="small muted" id="loadinfo"></span></div>
  </header>

  <section class="card" id="keybar">
    <div class="row">
      <label for="key">API key</label>
      <input id="key" type="password" size="28" autocomplete="off" placeholder="the key printed when the gateway started">
      <button class="primary" id="connect">Connect</button>
      <span class="small muted" id="keymsg"></span>
    </div>
  </section>

  <section class="card">
    <h2>Models</h2>
    <p class="small muted" style="margin:0 0 8px">One model is on the GPU at a time. Load unloads the current one first. A model that does not fit this GPU cannot be loaded; a disabled model cannot be loaded. Loading downloads the weights; to keep the disk from filling, only the last <span id="keep">2</span> used models stay on disk and older ones are deleted first.</p>
    <div class="table-wrap"><table>
      <thead><tr><th>Model</th><th class="num">GiB</th><th>Takes</th><th>State</th><th>On disk</th><th>Actions</th></tr></thead>
      <tbody id="models"></tbody>
    </table></div>
    <p class="small bad" id="modelerr" style="color: var(--bad)"></p>
  </section>

  <div class="grid2">
    <section class="card">
      <h2>Public endpoint (Cloudflare quick tunnel)</h2>
      <p class="small muted" style="margin:0 0 8px">For testing: no uptime guarantee, the URL changes whenever the tunnel restarts, 200 requests in flight at most, no streaming.
        Check the host's terms before serving from it (Google Colab disallows it).</p>
      <div class="row"><span class="chip" id="tstate">off</span><button id="tstart">Start tunnel</button><button id="tstop">Stop</button></div>
      <div class="row" style="margin-top:8px"><span class="mono" id="turl"></span><button id="tcopy" hidden>Copy</button></div>
      <div class="small muted" id="tinfo" style="margin-top:4px"></div>
      <pre id="curl" hidden></pre>
    </section>

    <section class="card">
      <h2>Response times</h2>
      <div class="table-wrap"><table>
        <thead><tr><th>Model</th><th class="num">Calls</th><th class="num">Errors</th><th class="num">p50 ms</th><th class="num">p95 ms</th></tr></thead>
        <tbody id="stats"><tr><td colspan="5" class="muted">No calls yet.</td></tr></tbody>
      </table></div>
    </section>
  </div>

  <section class="card">
    <h2>Try the loaded model</h2>
    <div class="tabs" role="tablist" aria-label="Input type">
      <button role="tab" aria-selected="true" data-tab="text">Text</button>
      <button role="tab" aria-selected="false" data-tab="image">Image</button>
      <button role="tab" aria-selected="false" data-tab="video">Video</button>
    </div>
    <div id="tab-image" hidden><input type="file" id="imgfile" accept="image/*"></div>
    <div id="tab-video" hidden><div class="row"><input type="file" id="vidfile" accept="video/*"><label>frames <input id="nframes" type="number" min="1" max="16" value="6" style="width:60px"></label></div>
      <p class="small muted" style="margin:4px 0 0">The server takes images, so the video is sampled into evenly spaced frames in your browser.</p></div>
    <label class="small muted" for="state-text">State</label>
    <textarea id="state-text" rows="2">Our checkout started returning errors and orders are blocked.</textarea>
    <label class="small muted" for="questions">Questions (JSON)</label>
    <textarea id="questions" rows="9"></textarea>
    <div class="row" style="margin-top:8px"><button class="primary" id="run">Run</button><button id="run5">Run 5 times</button><span class="small muted" id="runinfo"></span></div>
    <pre id="out" hidden></pre>
  </section>
</main>

<script>
"use strict";
const $ = (id) => document.getElementById(id);
const el = (tag, props = {}, ...kids) => {
  const n = document.createElement(tag);
  for (const [k, v] of Object.entries(props)) { if (k === "text") n.textContent = v; else if (k.startsWith("on")) n.addEventListener(k.slice(2), v); else if (v !== null && v !== undefined && v !== false) n.setAttribute(k, v === true ? "" : v); }
  kids.flat().forEach((c) => c != null && n.append(c));
  return n;
};
const chip = (text, tone = "") => el("span", { class: `chip ${tone}`, text });

let KEY = sessionStorage.getItem("jevgw-key") || "";
const hashKey = new URLSearchParams(location.hash.slice(1)).get("key");
if (hashKey) { KEY = hashKey; sessionStorage.setItem("jevgw-key", KEY); history.replaceState(null, "", location.pathname); }
let STATUS = null, BUSY = false, TAB = "text";

async function api(method, path, body) {
  const t0 = performance.now();
  const res = await fetch(path, { method, headers: { Authorization: `Bearer ${KEY}`, "Content-Type": "application/json" }, body: body === undefined ? undefined : JSON.stringify(body) });
  let json = null; try { json = await res.json(); } catch (_) { /* empty body */ }
  return { ok: res.ok, status: res.status, json, ms: performance.now() - t0, serverMs: res.headers.get("X-Latency-Ms") };
}

const QUESTIONS = {
  text: { team: { type: "choice", instructions: "Which team should handle this?", criteria: { billing: "Payments or invoices", technical: "Bugs or outages" } },
          urgency: { type: "score", instructions: "How urgent is it?", criteria: ["Can wait", "This week", "Today"] },
          outage: { type: "noul", instructions: "Is a service down?" } },
  image: { subject: { type: "noul", instructions: "Does this image contain a person?" }, setting: { type: "choice", instructions: "Is it indoors or outdoors?", criteria: { indoors: null, outdoors: null } } },
  video: { person: { type: "noul", instructions: "Is a person visible in these frames?" } },
};
function setTab(tab) {
  TAB = tab;
  document.querySelectorAll("[data-tab]").forEach((b) => b.setAttribute("aria-selected", String(b.dataset.tab === tab)));
  $("tab-image").hidden = tab !== "image"; $("tab-video").hidden = tab !== "video";
  $("questions").value = JSON.stringify(QUESTIONS[tab], null, 1);
  $("state-text").value = { text: "Our checkout started returning errors and orders are blocked.", image: "A photo.", video: "Frames sampled evenly from one video." }[tab];
}

function stateOf(m, s) {
  if (m.loaded) return chip("loaded", "good");
  if (s.loading === m.id) return chip("loading…", "warn");
  if (!m.runnable) return chip(m.status || "no recipe yet", "");
  if (!m.enabled) return chip("disabled", "");
  if (!m.fits) return chip(`needs ${m.gpu_hint || m.vram_gib + " GiB"}`, "bad");
  return chip("ready", "");
}
function renderModels(s) {
  $("models").replaceChildren(...s.models.map((m) => {
    const canLoad = m.runnable && m.enabled && m.fits && !BUSY && !m.loaded && !s.loading;
    const takes = [m.vision ? "text, images" : "text"].join("");
    return el("tr", { class: m.loaded ? "loaded" : "" },
      el("td", {}, el("strong", { text: m.name }), el("div", { class: "small muted mono", text: m.id }),
        m.note ? el("details", {}, el("summary", { class: "small muted", text: "About" }), el("div", { class: "small muted", text: m.note }),
          m.verified ? el("div", { class: "small muted", text: `Verified: ${m.verified}` }) : null) : null),
      el("td", { class: "num", text: m.vram_gib ? String(m.vram_gib) : "CPU" }),
      el("td", { text: takes }),
      el("td", {}, stateOf(m, s)),
      el("td", { class: "small", text: !m.disk_gib ? "n/a" : m.cached ? "yes" : `no (${m.disk_gib} GiB download)` }),
      el("td", {}, el("div", { class: "row" },
        m.loaded ? el("button", { onclick: () => act("unload"), disabled: BUSY, text: "Unload" })
                 : el("button", { class: "primary", onclick: () => act("load", m.id), disabled: !canLoad, text: "Load" }),
        m.runnable ? el("button", { onclick: () => act("enable", m.id, !m.enabled), disabled: BUSY, text: m.enabled ? "Disable" : "Enable" }) : null,
        m.cached && !m.loaded ? el("button", { onclick: () => act("purge", m.id), disabled: BUSY, text: "Delete files" }) : null)));
  }));
}

async function act(kind, id, flag) {
  $("modelerr").textContent = "";
  BUSY = true; if (STATUS) renderModels(STATUS);
  const req = { load: () => api("POST", "/admin/select", { model: id, wait: false }),  // returns at once; the table shows progress unload: () => api("POST", "/admin/unload", {}),
                enable: () => api("POST", "/admin/enable", { model: id, enabled: flag }),
                purge: () => api("POST", "/admin/purge", { model: id }) }[kind];
  try {
    const r = await req();
    if (!r.ok) $("modelerr").textContent = (r.json && r.json.error) || `HTTP ${r.status}`;
  } catch (e) { $("modelerr").textContent = String(e); }
  BUSY = false; refresh();
}

function renderHeader(s) {
  $("gpu").textContent = s.gpu ? `${s.gpu}, ${s.gpu_gib} GiB` : "no GPU";
  const d = s.disk; $("disk").textContent = d ? `disk: ${d.free_gib} of ${d.total_gib} GiB free` : "disk: not managed"; $("disk").className = "chip " + (d && d.free_gib < 8 ? "warn" : "");
  if (d) $("keep").textContent = d.keep;
  const st = $("state"); st.className = "chip " + (s.loaded ? "good" : s.loading ? "warn" : "");
  st.textContent = s.loaded ? `loaded: ${s.loaded}` : s.loading ? `loading ${s.loading}…` : "no model loaded";
  const pr = s.progress;
  $("loadinfo").textContent = pr ? (pr.phase === "downloading" ? `downloading: ${pr.done_gib} of ${pr.total_gib} GiB` : "starting the model server (the first start can take several minutes)")
    : s.loaded && s.load_s != null ? `loaded in ${s.load_s} s, warm-up ${s.warm_ms ?? "?"} ms` : s.error ? `last load failed: ${s.error.slice(0, 160)}. Press Load to retry.` : "";
}

function renderTunnel(t) {
  const on = t.running; $("tstate").textContent = on ? "on" : t.available === false ? "not available" : "off"; $("tstate").className = "chip " + (on ? "good" : "");
  $("turl").textContent = on ? t.url : ""; $("tcopy").hidden = !on;
  $("tstart").disabled = on || t.available === false; $("tstop").disabled = !on;
  $("tinfo").textContent = on ? `up ${t.up_s}s, restarted ${t.restarts}×` : t.error ? `last error: ${t.error}` : "";
  const c = $("curl"); c.hidden = !on;
  if (on) c.textContent = `curl ${t.url}/v1/systemone \\\n  -H "Authorization: Bearer $KEY" -H "Content-Type: application/json" \\\n  -d '{"state": "Our checkout is returning errors.", "questions": {"outage": {"type": "noul", "instructions": "Is a service down?"}}}'`;
}

function renderStats(stats) {
  const rows = Object.entries(stats);
  $("stats").replaceChildren(...(rows.length ? rows.map(([m, v]) => el("tr", {}, el("td", { class: "mono", text: m }), el("td", { class: "num", text: String(v.requests) }),
    el("td", { class: "num", text: String(v.errors) }), el("td", { class: "num", text: String(v.p50_ms) }), el("td", { class: "num", text: String(v.p95_ms) })))
    : [el("tr", {}, el("td", { colspan: 5, class: "muted", text: "No calls yet." }))]));
}

let timer = null;
async function refresh() {
  clearTimeout(timer);
  try {
    const [m, t, st] = await Promise.all([api("GET", "/v1/models"), api("GET", "/v1/tunnel"), api("GET", "/v1/stats")]);
    if (m.status === 401) { $("state").textContent = "enter the API key"; $("state").className = "chip bad"; $("keymsg").textContent = KEY ? "that key was refused" : ""; return; }
    $("keymsg").textContent = ""; STATUS = m.json;
    renderHeader(m.json); renderModels(m.json); renderTunnel(t.json); renderStats(st.json.stats);
    $("version").textContent = `v${st.json.version}`;
  } catch (e) { $("state").textContent = "gateway unreachable"; $("state").className = "chip bad"; }
  timer = setTimeout(refresh, STATUS && STATUS.loading ? 1500 : 4000);
}

async function frames(file, n) {
  const url = URL.createObjectURL(file), v = document.createElement("video");
  v.muted = true; v.src = url; await new Promise((ok, no) => { v.onloadedmetadata = ok; v.onerror = () => no(new Error("cannot read that video")); });
  const c = document.createElement("canvas"), scale = Math.min(1, 640 / v.videoWidth); c.width = v.videoWidth * scale; c.height = v.videoHeight * scale;
  const out = [];
  for (let i = 0; i < n; i++) {
    v.currentTime = v.duration * (i + 0.5) / n; await new Promise((ok) => { v.onseeked = ok; });
    c.getContext("2d").drawImage(v, 0, 0, c.width, c.height); out.push(c.toDataURL("image/jpeg", 0.85));
  }
  URL.revokeObjectURL(url); return out;
}
const toDataUrl = (file) => new Promise((ok, no) => { const r = new FileReader(); r.onload = () => ok(r.result); r.onerror = no; r.readAsDataURL(file); });

async function buildBody() {
  const body = { state: $("state-text").value, questions: JSON.parse($("questions").value) };
  if (TAB === "image") { const f = $("imgfile").files[0]; if (!f) throw new Error("choose an image"); body.images = [await toDataUrl(f)]; }
  if (TAB === "video") { const f = $("vidfile").files[0]; if (!f) throw new Error("choose a video"); body.images = await frames(f, Math.max(1, Math.min(16, +$("nframes").value || 6))); }
  return body;
}
async function run(times) {
  const out = $("out"); out.hidden = false; $("runinfo").textContent = "running…";
  try {
    const body = await buildBody(), ms = [], server = []; let last = null;
    for (let i = 0; i < times; i++) { last = await api("POST", "/v1/systemone", body); ms.push(last.ms); if (last.serverMs) server.push(+last.serverMs); if (!last.ok) break; }
    const med = (a) => a.length ? [...a].sort((x, y) => x - y)[a.length >> 1] : null;
    $("runinfo").textContent = `HTTP ${last.status} · round trip ${Math.round(med(ms))} ms${server.length ? ` · model ${Math.round(med(server))} ms` : ""}${times > 1 ? ` (median of ${ms.length})` : ""}`;
    out.textContent = JSON.stringify(last.json, null, 1);
  } catch (e) { out.textContent = String(e); $("runinfo").textContent = "failed"; }
  refresh();
}

document.querySelectorAll("[data-tab]").forEach((b) => b.addEventListener("click", () => setTab(b.dataset.tab)));
$("connect").addEventListener("click", () => { KEY = $("key").value.trim(); sessionStorage.setItem("jevgw-key", KEY); refresh(); });
$("key").addEventListener("keydown", (e) => { if (e.key === "Enter") $("connect").click(); });
$("run").addEventListener("click", () => run(1)); $("run5").addEventListener("click", () => run(5));
$("tstart").addEventListener("click", async () => { $("tstate").textContent = "starting…"; const r = await api("POST", "/admin/tunnel", { action: "start" }); if (!r.ok) $("tinfo").textContent = r.json && r.json.error; refresh(); });
$("tstop").addEventListener("click", async () => { await api("POST", "/admin/tunnel", { action: "stop" }); refresh(); });
$("tcopy").addEventListener("click", () => navigator.clipboard.writeText($("turl").textContent));
setTab("text"); if (KEY) $("key").value = KEY; refresh();
</script>
</body>
</html>

In [ ]:
%%writefile /content/gw/models.json
{
 "note": "Models that run on stock llama.cpp. vram_gib is the measured peak GPU memory (our RTX 5090, -ub 16384, with the vision projector).",
 "models": [
  {
   "id": "clef-flash-q4km",
   "name": "Clef Flash Q4_K_M",
   "kind": "llama",
   "repo": "bartowski/Cloudflare_clef-flash-GGUF",
   "file": "Cloudflare_clef-flash-Q4_K_M.gguf",
   "file_gib": 5.63,
   "mmproj": "mmproj-Cloudflare_clef-flash-f16.gguf",
   "mmproj_gib": 0.86,
   "vram_gib": 10.8,
   "vision": true,
   "gpu_hint": "T4 (free tier)",
   "note": "9B decision model, Q4_K_M (about 5.3 bits per weight). Within about 0.4 points of full precision on our tests.",
   "verified": "text, image and video-as-frames through the gateway (CPU only: the box's GPU was busy)"
  },
  {
   "id": "clef-flash-q2k",
   "name": "Clef Flash Q2_K",
   "kind": "llama",
   "repo": "bartowski/Cloudflare_clef-flash-GGUF",
   "file": "Cloudflare_clef-flash-Q2_K.gguf",
   "file_gib": 3.91,
   "mmproj": "mmproj-Cloudflare_clef-flash-f16.gguf",
   "mmproj_gib": 0.86,
   "vram_gib": 9.3,
   "vision": true,
   "gpu_hint": "T4 (free tier)",
   "note": "Q2_K (about 3.7 bits per weight). About 7 points below full precision on world knowledge, 3 on images.",
   "verified": "text, image and video-as-frames through the gateway (CPU only: the box's GPU was busy)"
  }
 ]
}

In [ ]:
%%writefile /content/gw/setup_llama.sh
#!/bin/bash
# Put a llama-server that supports Clef (build b11430 or newer) at $LLAMA_DIR/llama-server and print its path.
#
#   T4 (sm_75):  LLAMA_T4_URL=<url of llama-b11430-t4.tar.gz> ./setup_llama.sh    prebuilt with real sm_75 code
#   L4 / RTX 30-50 series:                    ./setup_llama.sh                    the official release already has native code for sm_86/89/120
#   anything else (A100 etc.) uses the official release, whose PTX the driver compiles on first start (slow once, needs a recent driver)
#   BUILD=1 ./setup_llama.sh                                                       build from source for this GPU: about 6 min on 20 cores, so about an hour on Colab's 2
set -euo pipefail
TAG=${TAG:-b11430}; LLAMA_DIR=${LLAMA_DIR:-$PWD/llama}; mkdir -p "$LLAMA_DIR"; cd "$LLAMA_DIR"
cc=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d . || true)
if [ -x "$LLAMA_DIR/bin/llama-server" ]; then echo "$LLAMA_DIR/bin/llama-server"; exit 0; fi
rm -rf "$LLAMA_DIR/bin"
# Everything is built in a scratch folder and moved into bin/ only when it is complete, so an interrupted run leaves nothing half-done
# and running this again is always safe.
tmp=$(mktemp -d "$LLAMA_DIR/.tmp.XXXXXX"); trap 'rm -rf "$tmp"' EXIT; mkdir -p "$tmp/bin"
if [ "${BUILD:-0}" = 1 ]; then
  export PATH=/usr/local/cuda/bin:$PATH CUDACXX=/usr/local/cuda/bin/nvcc
  [ -d src ] || git clone -q --depth 1 --branch "$TAG" https://github.com/ggml-org/llama.cpp src
  cmake -S src -B bld -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="${cc:-75}" -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release >/dev/null
  cmake --build bld --target llama-server -j"$(nproc)" >/dev/null
  cp bld/bin/llama-server bld/bin/*.so* "$tmp/bin/"
elif [ "$cc" = 75 ] && [ -n "${LLAMA_T4_URL:-}" ]; then
  curl -fsSL --retry 3 "$LLAMA_T4_URL" | tar xz -C "$tmp/bin"
else
  [ "$cc" = 75 ] && echo "note: T4 with no LLAMA_T4_URL: using the official release, whose T4 code is PTX the driver compiles on first start (slow once, needs a recent driver). Set LLAMA_T4_URL or BUILD=1." >&2
  curl -fsSL --retry 3 "https://github.com/ggml-org/llama.cpp/releases/download/$TAG/llama-$TAG-bin-ubuntu-cuda-12.8-x64.tar.gz" | tar xz -C "$tmp"
  mv "$tmp/llama-$TAG"/* "$tmp/bin/"
fi
[ -x "$tmp/bin/llama-server" ] || { echo "llama-server missing after install" >&2; exit 1; }
rm -rf bin; mv "$tmp/bin" bin
echo "$LLAMA_DIR/bin/llama-server"

## 2. Check the GPU

In [ ]:
import os, subprocess, sys
sys.path.insert(0, "/content/gw")
for name in [n for n in sys.modules if n == "jevgw" or n.startswith("jevgw.")]:
    del sys.modules[name]  # pick up the files written above, even if this notebook ran an older copy before
from jevgw import catalog

gpu_line = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout
if not gpu_line:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again.")
print(gpu_line)
gpu, mib = catalog.gpu_info()
for m in catalog.load("/content/gw/models.json").values():
    print(("fits   " if catalog.fits(m, mib) else "NO FIT ") + f"{m['id']:<17} {m['vram_gib']:>5} GiB GPU, {m['file_gib'] + m.get('mmproj_gib', 0):.1f} GiB download  {m['name']}")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

## 3. Install the llama.cpp server
Takes about a minute the first time. Run again if it fails: a broken install is removed and redone.

In [ ]:
LLAMA_T4_URL = ""  #@param {type:"string"}  # optional: a prebuilt T4 build (faster first start); see the README
from jevgw import notebook
LLAMA_BIN = notebook.ensure_llama(LLAMA_T4_URL)

## 4. Start the gateway and open the control panel
Run it as often as you like: a running gateway is reused (same key), a stuck one is replaced, a busy port is skipped. The panel loads, unloads, enables, disables and deletes models.

In [ ]:
KEEP = 2  #@param {type:"integer"}  # models kept on disk
API_KEY = ""  #@param {type:"string"}  # leave empty for a random key (it is printed below); or choose your own
client, info = notebook.start(LLAMA_BIN, KEEP, key=API_KEY)
KEY = info["key"]
notebook.show_panel(info)

## 5. Load a model and test it
The first load downloads the model and shows progress. If it fails it retries; run the cell again to try again. Or use the panel above.

In [ ]:
MODEL = "clef-flash-q4km"  #@param ["clef-flash-q4km", "clef-flash-q2k"]
notebook.wait_ready(client, MODEL)

In [ ]:
IMAGE = ""  #@param {type:"string"}  # optional: a path to your own image
VIDEO = ""  #@param {type:"string"}  # optional: a path to your own video (upload it in the Files panel)
import json
from jevgw import client as jc
rows = jc.demo(client, IMAGE or None, VIDEO or None, frames=6, n=5)  # five timed calls per test, after one warm-up
for r in rows:
    print(r["test"], "->", json.dumps(r["answers"])[:200])

## 6. Serve mode (opt-in): a public endpoint
**Read this first.** Colab's FAQ lists, for all runtimes, "web service offerings not related to interactive compute with Colab" and
"connecting to remote proxies", and for free runtimes "bypassing the notebook UI to interact primarily via a web UI". It also says
"Runtimes will time out if you are idle." A public endpoint is the kind of thing those lines describe, and the audio loop is a workaround for the
idle timeout. Google may disconnect the session or restrict your Colab use. It is your account and your call. For anything you want to keep running,
host the gateway on a GPU machine that allows serving (`python -m jevgw --tunnel`).

What this does: opens a free Cloudflare quick tunnel (testing only: no uptime promise, the URL changes if it restarts, 200 requests in flight at most,
no streaming), keeps it alive with a watchdog, and plays a silent audio loop. The URL serves the API only, not the panel, and every call needs the API key.

In [ ]:
SERVE = False  #@param {type:"boolean"}
if SERVE:
    tunnel = client.tunnel("start")
    sample = {"state": "Our checkout is returning errors.", "questions": {"outage": {"type": "noul", "instructions": "Is a service down?"}}}
    print("endpoint:", tunnel["url"])
    print("key     :", KEY)
    print(f"curl {tunnel['url']}/v1/systemone -H 'Authorization: Bearer {KEY}' -H 'Content-Type: application/json' -d '{json.dumps(sample)}'")
else:
    print("Serve mode is off.")

In [ ]:
# a silent audio loop: some people report it keeps the tab counted as active. Google does not document this, it can stop working,
# and it does not lift the 12 hour limit. Keep this tab open and in the foreground.
import base64, io, wave
from IPython.display import HTML, display
buf = io.BytesIO()
with wave.open(buf, "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(8000); w.writeframes(b"\x00\x00" * 8000)
display(HTML('<audio autoplay loop controls style="width:100%"><source src="data:audio/wav;base64,' + base64.b64encode(buf.getvalue()).decode() + '"></audio>'))

In [ ]:
# status: run again any time. The URL changes if the tunnel restarted.
print("tunnel:", client.tunnel())
s = client.models()
print("loaded:", s["loaded"], "| load", s["load_s"], "s | warm-up", s["warm_ms"], "ms | disk free", s["disk"]["free_gib"], "GiB")
for model, v in client.stats().items():
    print(f"{model:<18} {v['requests']:>5} calls, {v['errors']} errors, p50 {v['p50_ms']} ms, p95 {v['p95_ms']} ms")

## 7. Stop or reset
Leave this on `nothing` for Run all. **stop** frees the GPU and closes the tunnel. **reset** also clears any half-finished state (a stuck download or load); then Run all starts clean. Downloaded models are kept unless you pick the last option.

In [ ]:
ACTION = "nothing"  #@param ["nothing", "stop", "reset", "reset and delete downloaded models"]
if ACTION == "stop":
    notebook.stop()
    print("stopped. Run step 4 to start again.")
elif ACTION.startswith("reset"):
    notebook.reset(delete_models=ACTION.endswith("models"))
else:
    print("nothing to do")